# Round 3 results
After ProteinMPNN redesign from selected backbones

| Metric | Round1 | Round2 | Round2.5 | Round3 |
|---|---|---|---|---|
| Total designs | 2480 | 9520 | 59 | 2701 |
| ipAE < 5 | 2 (0.08%) | 88 (0.92%) | 22 (37.3%) | 98 (3.6%) |
| Pass all filters | 18 (0.73%) | 402 (4.2%) | 50 (87.4%) | 610 (22.6%) |
| Backbones with >=1 pass | 15/310 (4.8%) | 190/1190 (16.0%) | 50/59 (84.7%) | 610/2701 (22.6%) |

**What to do next**
With 610 passing designs:
1. Redo specificity scoring and compute AF3 stats
2. Combining passing designs from round 2, deep MPNN mining on top backbones — Running 32–50 sequences per backbone at T=0.1 and 0.2 on the top ~20–30 backbones (those with ≥3 passing sequences) would be the highest-yield next step.

## Calculate ipSAE

In [1]:
import sys
import glob
from pathlib import Path
from collections import OrderedDict
import pandas as pd
from scipy.stats import spearmanr

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 120

# Add the directory containing pmhci_ipsae.py and pae_cutoff_analysis.py
# to the path. Edit this if your scripts are elsewhere.
SCRIPT_DIR = Path(".")   # <-- update if needed
sys.path.insert(0, str(SCRIPT_DIR))

from calc_ipsae import (
    load_pae_matrix, build_chain_slices,
    chain_lengths_from_pdb, score_binder_peptide_ipsae, print_summary,
)

print("Imports OK")

Imports OK


In [2]:
PAE_DIR  = Path("/n/groups/marks/users/aaron/pmhc_cp/af_init_guess/outputs/r3/af2_kras/pae")
PDB_DIR  = Path("/n/groups/marks/users/aaron/pmhc_cp/af_init_guess/inputs/r3/threaded_pdbs")

NOMINAL_CUTOFF = 12.0   # Angstroms
DIST_CUTOFF    = 8.0    # Cα distance threshold for crystal contacts (Angstroms)
PEPTIDE_LEN = 9 # VVGADGVGK

In [3]:
# ── Discover design PAE + PDB pairs ─────────────────────────────────────────
npy_files = sorted(PAE_DIR.glob("*_pae.npy"))
pdb_files = []
for npy in npy_files:
    # Corresponding PDB: strip _pae.npy suffix, add .pdb
    stem = npy.stem.replace("_af2pred_pae", "")
    pdb  = PDB_DIR / f"{stem}.pdb"
    pdb_files.append(str(pdb) if pdb.exists() else None)

labels = [Path(n).stem.replace("_pae", "") for n in npy_files]

print(f"Found {len(npy_files)} PAE files")
print(f"  PDB matched: {sum(p is not None for p in pdb_files)} / {len(npy_files)}")
for npy, pdb, label in zip(npy_files, pdb_files, labels):
    pdb_ok = "✓" if pdb else "✗ (no PDB)"
    print(f"  {label:<50s} {pdb_ok}")

Found 2701 PAE files
  PDB matched: 2701 / 2701
  orient_252_pt20__31_pt12__0_a10_t0.1_s7_af2pred    ✓
  orient_252_pt20__31_pt12__0_a14_t0.15_s8_af2pred   ✓
  orient_252_pt20__31_pt12__0_a15_t0.15_s6_af2pred   ✓
  orient_252_pt20__31_pt12__0_a25_t0.2_s8_af2pred    ✓
  orient_252_pt20__31_pt12__0_a3_t0.1_s3_af2pred     ✓
  orient_252_pt20__31_pt12__0_a7_t0.1_s8_af2pred     ✓
  orient_252_pt20__31_pt12__0_a8_t0.1_s4_af2pred     ✓
  orient_252_pt20__31_pt12__0_a8_t0.1_s7_af2pred     ✓
  orient_252_pt20__31_pt12__14_a14_t0.15_s5_af2pred  ✓
  orient_252_pt20__31_pt12__14_a15_t0.15_s1_af2pred  ✓
  orient_252_pt20__31_pt12__14_a15_t0.15_s3_af2pred  ✓
  orient_252_pt20__31_pt12__14_a16_t0.15_s4_af2pred  ✓
  orient_252_pt20__31_pt12__14_a20_t0.15_s5_af2pred  ✓
  orient_252_pt20__31_pt12__14_a22_t0.2_s3_af2pred   ✓
  orient_252_pt20__31_pt12__14_a8_t0.1_s1_af2pred    ✓
  orient_252_pt20__31_pt12__14_a9_t0.1_s8_af2pred    ✓
  orient_252_pt20__31_pt12__16_a1_t0.1_s5_af2pred    ✓
  orient_252_pt20

In [4]:
# ── Quick ipSAE summary table at the nominal cutoff ─────────────────────────
print(f"{'Design':<50s} {'binder':>7} {'MHC':>5} {'pep':>4} {'ipSAE':>7} {'n_contacts':>11} {'mean_PAE':>9}")
print("─" * 100)

results_table = []
for npy, pdb, label in zip(npy_files, pdb_files, labels):
    try:
        cl = chain_lengths_from_pdb(pdb, peptide_length=PEPTIDE_LEN) if pdb else None
        if cl is None:
            print(f"  {label}: no PDB, skipping")
            continue
        r = score_binder_peptide_ipsae(str(npy), cl, pae_cutoff=NOMINAL_CUTOFF)
        results_table.append({"label": label, "chain_lengths": cl, **r})
        print(f"  {label:<48s} {cl['binder']:>7} {cl['MHC']:>5} {cl['peptide']:>4} "
              f"  {r['ipsae_binder_peptide']:>6.4f}  {r['n_contacts']:>10}  {r['mean_pae_bp']:>8.2f} Å")
    except Exception as e:
        print(f"  {label}: ERROR — {e}")

print(f"\nScored {len(results_table)} designs at PAE cutoff = {NOMINAL_CUTOFF} Å")

results_df = pd.DataFrame(results_table)

Design                                              binder   MHC  pep   ipSAE  n_contacts  mean_PAE
────────────────────────────────────────────────────────────────────────────────────────────────────
  orient_252_pt20__31_pt12__0_a10_t0.1_s7_af2pred       97   178    9   0.0000           0     22.10 Å
  orient_252_pt20__31_pt12__0_a14_t0.15_s8_af2pred      97   178    9   0.8607         853      4.20 Å
  orient_252_pt20__31_pt12__0_a15_t0.15_s6_af2pred      97   178    9   0.0000           0     20.35 Å
  orient_252_pt20__31_pt12__0_a25_t0.2_s8_af2pred       97   178    9   0.0000           0     22.40 Å
  orient_252_pt20__31_pt12__0_a3_t0.1_s3_af2pred        97   178    9   0.8721         868      3.76 Å
  orient_252_pt20__31_pt12__0_a7_t0.1_s8_af2pred        97   178    9   0.7129         843      6.61 Å
  orient_252_pt20__31_pt12__0_a8_t0.1_s4_af2pred        97   178    9   0.0000           0     21.56 Å
  orient_252_pt20__31_pt12__0_a8_t0.1_s7_af2pred        97   178    9   0.4528

  orient_252_pt20__31_pt12__73_a6_t0.1_s3_af2pred       97   178    9   0.0000           0     22.03 Å
  orient_252_pt20__31_pt12__73_a7_t0.1_s5_af2pred       97   178    9   0.0000           0     21.82 Å
  orient_252_pt20__31_pt12__73_a7_t0.1_s6_af2pred       97   178    9   0.0000           0     22.52 Å
  orient_252_pt20__31_pt12__73_a8_t0.1_s1_af2pred       97   178    9   0.0000           0     22.71 Å
  orient_252_pt20__31_pt12__91_a13_t0.15_s2_af2pred      97   178    9   0.0000           0     25.16 Å
  orient_252_pt20__31_pt12__91_a15_t0.15_s1_af2pred      97   178    9   0.3623         366     16.63 Å
  orient_252_pt20__31_pt12__91_a8_t0.1_s7_af2pred       97   178    9   0.0000           0     22.69 Å
  orient_252_pt20__31_pt12__97_a1_t0.1_s1_af2pred       97   178    9   0.8802         873      3.57 Å
  orient_252_pt20__31_pt12__97_a1_t0.1_s4_af2pred       97   178    9   0.0000           0     20.94 Å
  orient_252_pt20__31_pt12__97_a2_t0.1_s2_af2pred       97   178    9  

  orient_255_pt12__18_pt12__24_a1_t0.1_s6_af2pred       93   178    9   0.3985         448     11.96 Å
  orient_255_pt12__18_pt12__24_a2_t0.1_s2_af2pred       93   178    9   0.0190          34     17.67 Å
  orient_255_pt12__18_pt12__24_a2_t0.1_s4_af2pred       93   178    9   0.0000           0     22.00 Å
  orient_255_pt12__18_pt12__24_a2_t0.1_s5_af2pred       93   178    9   0.1946         169     14.63 Å
  orient_255_pt12__18_pt12__24_a2_t0.1_s6_af2pred       93   178    9   0.7283         806      6.26 Å
  orient_255_pt12__18_pt12__25_a1_t0.1_s1_af2pred       93   178    9   0.0000           0     25.39 Å
  orient_255_pt12__18_pt12__25_a1_t0.1_s4_af2pred       93   178    9   0.0000           0     25.54 Å
  orient_255_pt12__18_pt12__25_a1_t0.1_s5_af2pred       93   178    9   0.0000           0     24.06 Å
  orient_255_pt12__18_pt12__25_a1_t0.1_s7_af2pred       93   178    9   0.0000           0     21.05 Å
  orient_255_pt12__18_pt12__25_a2_t0.1_s1_af2pred       93   178    9   0

  orient_255_pt12__18_pt12__42_a1_t0.1_s4_af2pred       93   178    9   0.5661         685      9.32 Å
  orient_255_pt12__18_pt12__42_a1_t0.1_s5_af2pred       93   178    9   0.0000           0     21.87 Å
  orient_255_pt12__18_pt12__42_a1_t0.1_s7_af2pred       93   178    9   0.2938         293     13.60 Å
  orient_255_pt12__18_pt12__42_a1_t0.1_s8_af2pred       93   178    9   0.0000           0     22.17 Å
  orient_255_pt12__18_pt12__42_a2_t0.1_s1_af2pred       93   178    9   0.0000           0     21.62 Å
  orient_255_pt12__18_pt12__45_a10_t0.1_s5_af2pred      93   178    9   0.0000           0     21.10 Å
  orient_255_pt12__18_pt12__45_a11_t0.15_s8_af2pred      93   178    9   0.0000           0     21.49 Å
  orient_255_pt12__18_pt12__45_a13_t0.15_s2_af2pred      93   178    9   0.7132         806      6.52 Å
  orient_255_pt12__18_pt12__45_a2_t0.1_s6_af2pred       93   178    9   0.3132         324     13.18 Å
  orient_255_pt12__18_pt12__45_a3_t0.1_s3_af2pred       93   178    9  

  orient_255_pt12__18_pt12__68_a1_t0.1_s5_af2pred       93   178    9   0.0020           5     19.75 Å
  orient_255_pt12__18_pt12__68_a1_t0.1_s7_af2pred       93   178    9   0.6710         784      7.29 Å
  orient_255_pt12__18_pt12__68_a1_t0.1_s8_af2pred       93   178    9   0.6604         784      7.54 Å
  orient_255_pt12__18_pt12__68_a2_t0.1_s1_af2pred       93   178    9   0.7197         819      6.31 Å
  orient_255_pt12__18_pt12__68_a2_t0.1_s2_af2pred       93   178    9   0.7536         818      5.77 Å
  orient_255_pt12__18_pt12__69_a10_t0.1_s6_af2pred      93   178    9   0.4832         570     11.21 Å
  orient_255_pt12__18_pt12__69_a13_t0.15_s2_af2pred      93   178    9   0.7672         806      5.66 Å
  orient_255_pt12__18_pt12__69_a13_t0.15_s3_af2pred      93   178    9   0.0000           0     19.40 Å
  orient_255_pt12__18_pt12__69_a13_t0.15_s7_af2pred      93   178    9   0.7747         820      5.45 Å
  orient_255_pt12__18_pt12__69_a18_t0.15_s8_af2pred      93   178    9

...

  orient_255_pt12__8_pt25__27_a3_t0.1_s2_af2pred        93   178    9   0.0000           0     22.02 Å
  orient_255_pt12__8_pt25__29_a10_t0.1_s1_af2pred       93   178    9   0.0000           0     22.70 Å
  orient_255_pt12__8_pt25__29_a10_t0.1_s2_af2pred       93   178    9   0.0000           0     22.28 Å
  orient_255_pt12__8_pt25__29_a11_t0.15_s8_af2pred      93   178    9   0.0000           0     22.28 Å
  orient_255_pt12__8_pt25__29_a13_t0.15_s7_af2pred      93   178    9   0.0000           0     22.70 Å
  orient_255_pt12__8_pt25__29_a2_t0.1_s7_af2pred        93   178    9   0.0000           0     21.74 Å
  orient_255_pt12__8_pt25__29_a4_t0.1_s1_af2pred        93   178    9   0.0000           0     22.45 Å
  orient_255_pt12__8_pt25__29_a6_t0.1_s8_af2pred        93   178    9   0.0353          47     16.49 Å
  orient_255_pt12__8_pt25__29_a9_t0.1_s3_af2pred        93   178    9   0.0345          46     16.67 Å
  orient_255_pt12__8_pt25__30_a2_t0.1_s6_af2pred        93   178    9   0

  orient_255_pt12__8_pt25__45_a2_t0.1_s7_af2pred        93   178    9   0.0000           0     22.66 Å
  orient_255_pt12__8_pt25__45_a2_t0.1_s8_af2pred        93   178    9   0.0000           0     23.11 Å
  orient_255_pt12__8_pt25__45_a3_t0.1_s1_af2pred        93   178    9   0.0000           0     21.30 Å
  orient_255_pt12__8_pt25__46_a1_t0.1_s3_af2pred        93   178    9   0.0000           0     22.60 Å
  orient_255_pt12__8_pt25__46_a1_t0.1_s5_af2pred        93   178    9   0.0000           0     22.80 Å
  orient_255_pt12__8_pt25__46_a2_t0.1_s1_af2pred        93   178    9   0.0323          43     17.36 Å
  orient_255_pt12__8_pt25__46_a2_t0.1_s2_af2pred        93   178    9   0.0000           0     22.95 Å
  orient_255_pt12__8_pt25__46_a2_t0.1_s3_af2pred        93   178    9   0.1471         138     15.20 Å
  orient_255_pt12__8_pt25__46_a2_t0.1_s5_af2pred        93   178    9   0.0000           0     21.32 Å
  orient_255_pt12__8_pt25__46_a2_t0.1_s6_af2pred        93   178    9   0

  orient_255_pt12__91_pt25__21_a21_t0.2_s2_af2pred      93   178    9   0.0020          13     18.47 Å
  orient_255_pt12__91_pt25__21_a23_t0.2_s2_af2pred      93   178    9   0.0000           0     23.16 Å
  orient_255_pt12__91_pt25__25_a1_t0.1_s2_af2pred       93   178    9   0.0019           5     19.09 Å
  orient_255_pt12__91_pt25__25_a1_t0.1_s3_af2pred       93   178    9   0.0000           0     23.62 Å
  orient_255_pt12__91_pt25__25_a1_t0.1_s4_af2pred       93   178    9   0.0000           0     21.51 Å
  orient_255_pt12__91_pt25__25_a1_t0.1_s6_af2pred       93   178    9   0.0021           7     19.15 Å
  orient_255_pt12__91_pt25__25_a1_t0.1_s7_af2pred       93   178    9   0.0000           0     21.79 Å
  orient_255_pt12__91_pt25__25_a1_t0.1_s8_af2pred       93   178    9   0.0018          12     17.91 Å
  orient_255_pt12__91_pt25__25_a2_t0.1_s7_af2pred       93   178    9   0.0000           0     22.32 Å
  orient_255_pt12__91_pt25__25_a2_t0.1_s8_af2pred       93   178    9   0

  orient_255_pt12__91_pt25__47_a2_t0.1_s5_af2pred       93   178    9   0.0000           0     21.28 Å
  orient_255_pt12__91_pt25__47_a2_t0.1_s6_af2pred       93   178    9   0.0000           0     22.99 Å
  orient_255_pt12__91_pt25__47_a3_t0.1_s1_af2pred       93   178    9   0.0020           3     21.03 Å
  orient_255_pt12__91_pt25__47_a3_t0.1_s3_af2pred       93   178    9   0.0000           0     22.43 Å
  orient_255_pt12__91_pt25__49_a1_t0.1_s1_af2pred       93   178    9   0.0000           0     22.23 Å
  orient_255_pt12__91_pt25__49_a1_t0.1_s3_af2pred       93   178    9   0.0000           0     22.97 Å
  orient_255_pt12__91_pt25__49_a1_t0.1_s4_af2pred       93   178    9   0.0000           0     22.75 Å
  orient_255_pt12__91_pt25__49_a1_t0.1_s5_af2pred       93   178    9   0.0000           0     23.55 Å
  orient_255_pt12__91_pt25__49_a1_t0.1_s6_af2pred       93   178    9   0.0000           0     21.94 Å
  orient_255_pt12__91_pt25__49_a1_t0.1_s7_af2pred       93   178    9   0

  orient_34_pt12__67_pt12__52_a5_t0.1_s5_af2pred        93   178    9   0.0000           0     22.43 Å
  orient_34_pt12__67_pt12__52_a6_t0.1_s6_af2pred        93   178    9   0.0020           5     20.49 Å
  orient_34_pt12__67_pt12__52_a9_t0.1_s1_af2pred        93   178    9   0.0000           0     24.26 Å
  orient_34_pt12__67_pt12__57_a11_t0.15_s4_af2pred      93   178    9   0.0000           0     27.32 Å
  orient_34_pt12__67_pt12__57_a18_t0.15_s5_af2pred      93   178    9   0.0000           0     25.46 Å
  orient_34_pt12__67_pt12__57_a22_t0.2_s2_af2pred       93   178    9   0.0000           0     26.70 Å
  orient_34_pt12__67_pt12__57_a22_t0.2_s8_af2pred       93   178    9   0.0000           0     23.35 Å
  orient_34_pt12__67_pt12__57_a23_t0.2_s5_af2pred       93   178    9   0.0000           0     27.40 Å
  orient_34_pt12__67_pt12__57_a25_t0.2_s4_af2pred       93   178    9   0.0000           0     26.55 Å
  orient_34_pt12__67_pt12__57_a8_t0.1_s4_af2pred        93   178    9   0

In [5]:
results_df[results_df['n_contacts']>0].sort_values(by='ipsae_binder_peptide', ascending=False)

,label,chain_lengths,ipsae_binder_peptide,ipsae_bp,ipsae_pb,n_contacts,d0,mean_pae_bp,mean_pae_contact
1431,orient_255_pt12__33_pt12__81_a4_t0.1_s5_af2pred,"{'binder': 93, 'MHC': 178, 'peptide': 9}",0.8959,0.8959,0.6658,824,9.754,3.421,3.204
1414,orient_255_pt12__33_pt12__7_a4_t0.1_s4_af2pred,"{'binder': 93, 'MHC': 178, 'peptide': 9}",0.8940,0.8940,0.7512,820,9.735,3.523,3.207
28,orient_252_pt20__31_pt12__27_a7_t0.1_s4_af2pred,"{'binder': 97, 'MHC': 178, 'peptide': 9}",0.8846,0.8846,0.6154,860,9.923,3.645,3.471
2019,orient_255_pt12__70_pt12__37_a1_t0.1_s8_af2pred,"{'binder': 93, 'MHC': 178, 'peptide': 9}",0.8846,0.8846,0.6648,837,9.816,3.435,3.435
1256,orient_255_pt12__33_pt12__26_a28_t0.2_s6_af2pred,"{'binder': 93, 'MHC': 178, 'peptide': 9}",0.8833,0.8833,0.6840,826,9.764,3.575,3.431
...,...,...,...,...,...,...,...,...,...
1133,orient_255_pt12__31_pt12__65_a1_t0.1_s7_af2pred,"{'binder': 93, 'MHC': 178, 'peptide': 9}",0.0017,0.0017,0.0000,1,0.500,20.466,11.945
192,orient_255_pt12__18_pt12__29_a28_t0.2_s3_af2pred,"{'binder': 93, 'MHC': 178, 'peptide': 9}",0.0017,0.0017,0.0000,1,0.500,21.376,11.969
1452,orient_255_pt12__33_pt12__88_a3_t0.1_s5_af2pred,"{'binder': 93, 'MHC': 178, 'peptide': 9}",0.0017,0.0017,0.0000,1,0.500,21.491,11.963
1578,orient_255_pt12__46_pt12__26_a2_t0.1_s2_af2pred,"{'binder': 93, 'MHC': 178, 'peptide': 9}",0.0017,0.0017,0.0000,1,0.500,21.393,11.944


In [6]:
len(results_df[results_df['n_contacts']>0].sort_values(by='ipsae_binder_peptide', ascending=False))

1281

In [7]:
ipae_df = pd.read_csv('/n/groups/marks/users/aaron/pmhc_cp/af_init_guess/outputs/r3/af2_kras_all_scores.csv')
merged_df = pd.merge(results_df[results_df['n_contacts']>0], ipae_df, how='left', left_on='label', right_on='description')
merged_df = merged_df[['label', 'ipsae_binder_peptide', 'ipsae_bp', 'ipsae_pb', 'n_contacts', 'd0', 'mean_pae_bp', 'mean_pae_contact', 'binder_aligned_rmsd', 'target_aligned_rmsd', 'pae_interaction', 'plddt_binder', 'plddt_target']]
pd.set_option('display.max_rows', None)
merged_df[merged_df['pae_interaction']<5].sort_values(by='ipsae_binder_peptide', ascending=False)

,label,ipsae_binder_peptide,ipsae_bp,ipsae_pb,n_contacts,d0,mean_pae_bp,mean_pae_contact,binder_aligned_rmsd,target_aligned_rmsd,pae_interaction,plddt_binder,plddt_target
770,orient_255_pt12__33_pt12__81_a4_t0.1_s5_af2pred,0.8959,0.8959,0.6658,824,9.754,3.421,3.204,0.357,1.511,3.859,95.889,95.594
755,orient_255_pt12__33_pt12__7_a4_t0.1_s4_af2pred,0.8940,0.8940,0.7512,820,9.735,3.523,3.207,0.700,1.556,4.085,94.430,95.992
15,orient_252_pt20__31_pt12__27_a7_t0.1_s4_af2pred,0.8846,0.8846,0.6154,860,9.923,3.645,3.471,0.602,0.782,3.982,95.205,95.177
1035,orient_255_pt12__70_pt12__37_a1_t0.1_s8_af2pred,0.8846,0.8846,0.6648,837,9.816,3.435,3.435,0.511,1.480,4.026,95.579,95.413
660,orient_255_pt12__33_pt12__26_a28_t0.2_s6_af2pred,0.8833,0.8833,0.6840,826,9.764,3.575,3.431,0.555,2.287,4.255,94.835,95.378
753,orient_255_pt12__33_pt12__7_a2_t0.1_s1_af2pred,0.8831,0.8831,0.6908,836,9.811,3.452,3.442,0.689,1.596,4.168,95.332,95.631
732,orient_255_pt12__33_pt12__67_a3_t0.1_s8_af2pred,0.8818,0.8818,0.6824,834,9.802,3.517,3.484,0.631,1.339,4.150,94.830,95.453
13,orient_252_pt20__31_pt12__27_a2_t0.1_s1_af2pred,0.8813,0.8813,0.6475,870,9.969,3.592,3.558,0.562,1.230,4.212,94.756,95.167
32,orient_252_pt20__31_pt12__97_a1_t0.1_s1_af2pred,0.8802,0.8802,0.6231,873,9.983,3.569,3.569,0.494,0.835,4.073,94.890,94.950
1059,orient_255_pt12__70_pt12__46_a1_t0.1_s2_af2pred,0.8784,0.8784,0.6938,816,9.716,3.869,3.501,0.753,1.214,4.436,94.216,95.369


In [8]:
merged_df.to_csv('/n/groups/marks/users/aaron/pmhc_cp/af_init_guess/outputs/r3/af2_ipae_ipsae_scores.csv')